In [23]:
#========================================================================
# LOGIC MAP
'''

AGSEC panel:


AGSEC2 is Parcel ID (ownership= 1,2,3,4..)(user rights=21,22,23..)
AGSEC3 is Both Parcel IDs + plot IDs for both (parcel = 1 plot = 1,2,3, Parcel 21 = 1,2,3 ). SEC3 also has 2 visits which need to transcribed into season 1 or 2 based on what they were that year.
AGSEC4 is the same but crop ID for plot ID also. Also 2 visits
AGSEC5 is the same as 4. also 2 visits.

Comment: We don't care about user rights/ownership here, they just have different numbering. When we take averages we do it across all plots/parcels. What is important is differing visits.


Task:

HH to crop level:
For each crop code:
- dummy_planted_yes_no
- sum all areas of all plots planted on that crop
- sum all harvest of all plots of that crop
- convert into calories (comes later, for now just kg)
- share of total land planted for this crop
- share of total calories for this crop

+ Parcel info from AGSEC2 (,A2aq19,A2aq20, ---> q19: dummy for each parcel = 1 if "1" or "2" or missing, =0 if "3". Then take average for all parcels belonging to HHID and create variable "A2AQ19_average".  q20: create dummy = 1 if "2" = 0 if "1" or "3" or missing for each parcel and take average. Name variable A2Q20_average_rainfed".

+ Plot info from sec3: A3Q4, 14,26: dummies taking 1 or 2. For each plot ID create dummies = 1 if "1" otherwise 0. Then for all plots belonging to HH take average for each question and create three dummies: A3Q4_average_use, A3Q14_average_use_x, A3Q26_average_use.
 A3Q38 = use value at plot level and aggregate for all plots and take average to add to HHID as A3AQ38_average_.
 Then A3Q39 is a number, same principle there, take average for all plots/parcels and add to HHID as A3Q39_average.
 A3Q41 is a dummy, here we set a new dummy = 1 if any Q41 =1 for that HHID. Name for new dummy A3Q41_any_labor
 A3Q43 is a value/number. Here just sum to HHID level.


AGSEC4:  Here we have crop codes (q6) and cropping system (q7). If q7 = 2 then q9 asks percentage of crop code planted on plot id. question 8 asks total acres of plot id planted.

Agsec5: question 6a: quantity of crop harvested that visit
6b: condition of harvest, 6c: unit code, 6d: conversion factor into kg.
question 15 ask how much of harvest was saved for seed for that crop --> sum across all plots to crop level and make dummies for each crop code. Same with question 16: lost or wasted harvest.

My idea to finish task + add the other dummies:

Start by creating dummies for all crop codes found in dataset. Assign = 1 if that HHID has that cropcode for any rows of "CROP_ID"
Then use A4AQ8 to calculate for total area of crop code planted (sum all plots belonging to HHID) and create area_cropcode_visit_X. Sum harvest same principle but use A5QAQ6A*6D and sum for each crop code. Create variable: cropcode_total_harvest_visit_X_KG.

Use A2AQ4 (parcel size) and sum all parcels belonging to HHID. Use this as total area and create variable: total_area, then use total_area_visit_X and divide by area_cropcode_visit_X to get share_cropcode_area_visit_X


GSEC11:

Keep H11Q01 as is, assign the number reported to the HHID for that row.

H11Q03 is an income code index with possible values: 11,12,13, 21,22,23, 31-36, 41,42,43,43,45
H11AQ04 is is dummy for income code ---> Create dummies called H11Q03(11), H11Q03(12) and so on.

H11AQ05 and 06 is value variables for income codes --> add them together for each row/income as new variables: H11AQ_inc_value(Income index value)


GSEC15C:

variable: ITEM_CODE = Codes/index: 301-311, 451-459, 461-469, 501-505, 601-605. ---> Create variables wide for each code, called: H15C(301) and so on. We will calculate value for each code down below and that should be filled for each income code.

H15CQ3: Unit of quantity for codes that have quantity
H15CQ4, 6, 8 are quantity columns for those codes that have quantity
H15CQ5, 7, 9 are value columns for codes

H15CQ10 is "unit price" so for codes that have no value recorded, use Q3*Q10*(Q4 + Q6 + Q8) to derive value for these. So if Q5 empty, Q7 filled and Q9 empty while Q4,6,8 are filled, use Q3*Q10*(Q4 + Q8) only, since Q7 value already filled. Then add together the derived value + Value at Q7 = Value for that Income code variable.

ALSO: In wave 7 and 8 some income codes are split up: NNN_N, create separate variables for these



GSEC16:

H16Q00 -Shock codes: 101-118 --> Create 18 dummies called H16Q1(101), H16Q1(102)... taking value 1 if H16Q1 = 1 for that row/shock index, otherwise 0. All rows in the original file have Q1 = 1.
-Then create H16Q2A(101) variables aswell and write over the value in Q2A (which is 1-12)
-Same for H16Q2B --> leave empty if cell empty
- H16Q3A-D ---> dummies for all shock codes: H16Q3A(NNN): 1 if Q3A: 1 for that shock, otherwise 0
-Same for H16Q4A-C but with numbers. Possible numbers: 1-15 and 96. Assign recorded number to correct shock.



GSEC18:

H18Q1(index): A,B,C,D + H18Q2(dummy): 1,1,1,1 -> H18Q1A = 1,H18Q1B = 1,H18Q1C = 1,H18Q1D = 1 (Dummies)
H18Q4: If any number reported for letters in Q1 --> H18Q4A = that number,H18Q4B = that number,H18Q4C = that number,H18Q4D = that number
H18Q5: Letter index dummy ---> H18Q5A: 1,H18Q5B: 1,H18Q5B: 1,H18Q5B: 1,

H18Q9(index): A,B,C,D,E,F + H18Q10(dummy): 1,1,1,1,1,1 ---> H18Q9A = 1,H18Q9B = 1,H18Q9C = 1,H18Q9D = 1 (Dummies)
H18Q11 (choice: 1-5 for A-F index) --> Same principle create Q11A,B,C... and assign the number reported for that A-F index letter.

'''
#========================================================================


'\n\nAGSEC panel:\n\n\nAGSEC2 is Parcel ID (ownership= 1,2,3,4..)(user rights=21,22,23..)\nAGSEC3 is Both Parcel IDs + plot IDs for both (parcel = 1 plot = 1,2,3, Parcel 21 = 1,2,3 ). SEC3 also has 2 visits which need to transcribed into season 1 or 2 based on what they were that year.\nAGSEC4 is the same but crop ID for plot ID also. Also 2 visits\nAGSEC5 is the same as 4. also 2 visits.\n\nTask:\n\nHH to crop level:\nFor each crop code:\n- dummy_planted_yes_no\n- sum all areas of all plots planted on that crop\n- sum all harvest of all plots of that crop\n- convert into calories (comes later, for now just kg)\n- share of total land planted for this crop\n- share of total calories for this crop\n\n+ Parcel info from AGSEC2 (,A2aq19,A2aq20, ---> q19: dummy for each parcel = 1 if "1" or "2" or missing, =0 if "3". Then take average for all parcels belonging to HHID and create variable "A2AQ19_average".  q20: create dummy = 1 if "2" = 0 if "1" or "3" or missing for each parcel and take av

In [1]:
from pathlib import Path
import pandas as pd

#=========================================================================
#Add files to use here

ROOT = Path(r"/")

BASE_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "All HHID wavelevel.xlsx"
)

GSEC9_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC9_standardized.csv"
)

GSEC10_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC10_standardized.csv"
)

GSEC12_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC12_standardized.csv"
)

GSEC13_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC13_WAVE1_standardized.csv"
)

GSEC17_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC17_standardized.csv"
)


#============================================================================

# Helpers
def clean_hhid(x):

    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x


def clean_wave(x):

    if pd.isna(x):
        return pd.NA

    x = str(x).strip().lower()
    x = x.replace("wave", "").replace("_", "").replace("-", "").strip()

    return int(float(x))


def load_table_clean(path, sheet_name=0):

    path = Path(path)

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name, dtype=str)

    elif path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=str)

    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")


def standardize_keys(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
    )

    rename_map = {}
    for col in df.columns:
        if col.lower() == "wave":
            rename_map[col] = "Wave"
        elif col.lower() == "hhid":
            rename_map[col] = "HHID"
        elif col.lower() == "hh_id_obs":
            rename_map[col] = "hh_id_obs"

    df = df.rename(columns=rename_map)

    if "Wave" not in df.columns:
        raise ValueError("Missing key column: Wave")

    if "HHID" not in df.columns:
        raise ValueError("Missing key column: HHID")

    df["Wave"] = df["Wave"].apply(clean_wave)
    df["HHID"] = df["HHID"].apply(clean_hhid)

    return df


def check_unique_keys(df, keys, name):

    dupes = df[df.duplicated(keys, keep=False)]

    if len(dupes) > 0:
        print(f"WARNING: {name} has duplicate keys on {keys}.")
        print(f"Number of duplicated rows: {len(dupes)}")
        display(dupes.sort_values(keys).head(20))
    else:
        print(f"{name}: keys are unique.")



In [2]:
### Load base and merge GSEC9

base = load_table_clean(BASE_FILE)
base = standardize_keys(base)

base = base[["Wave", "HHID", "hh_id_obs"]].copy()

check_unique_keys(base, ["Wave", "HHID"], "HHID base")

print("Base shape:", base.shape)
display(base.head())

gsec9 = load_table_clean(GSEC9_FILE)
gsec9 = standardize_keys(gsec9)

check_unique_keys(gsec9, ["Wave", "HHID"], "GSEC9")

print("GSEC9 shape:", gsec9.shape)
display(gsec9.head())

key_cols = ["Wave", "HHID"]

gsec9_renamed = gsec9.rename(
    columns={
        col: f"GSEC9__{col}"
        for col in gsec9.columns
        if col not in key_cols
    }
)

hh_panel = base.merge(
    gsec9_renamed,
    on=["Wave", "HHID"],
    how="left",
    validate="1:1"
)

print("Merged panel shape:", hh_panel.shape)
print("Rows in base:", len(base))
print("Rows after merge:", len(hh_panel))

gsec9_vars = [c for c in hh_panel.columns if c.startswith("GSEC9__")]
hh_panel["has_GSEC9"] = hh_panel[gsec9_vars].notna().any(axis=1)

print(hh_panel["has_GSEC9"].value_counts(dropna=False))

display(hh_panel.head())

HHID base: keys are unique.
Base shape: (21239, 3)


,Wave,HHID,hh_id_obs
0,1,1013000201,7000043
1,1,1013000204,7000045
2,1,1013000206,7000046
3,1,1013000210,7000047
4,1,1013000213,7000049


GSEC9: keys are unique.
GSEC9 shape: (21168, 28)


,Wave,SOURCE_FILE,SOURCE_SECTION,HHID,H9Q1,H9Q2,H9Q3,H9Q4,H9Q5,H9Q6,...,H9Q13,H9Q14,H9Q16,H9Q17,H9Q18,H9Q19,H9Q20,H9Q21,H9Q22,H9Q23
0,1,GSEC9.dta,GSEC9,1013000201,1.0,1.0,2.0,4.0,3.0,1.0,...,NaN,NaN,9.0,2.0,2.0,1.0,3.0,2.0,5.0,1.0
1,1,GSEC9.dta,GSEC9,1013000204,1.0,1.0,1.0,4.0,3.0,1.0,...,NaN,NaN,2.0,2.0,2.0,1.0,2.0,5.0,2.0,1.0
2,1,GSEC9.dta,GSEC9,1013000206,2.0,5.0,1.0,4.0,6.0,3.0,...,1.0,1500.0,9.0,2.0,2.0,1.0,2.0,5.0,4.0,1.0
3,1,GSEC9.dta,GSEC9,1013000210,1.0,1.0,2.0,4.0,3.0,1.0,...,NaN,NaN,1.0,8.0,NaN,NaN,1.0,5.0,2.0,1.0
4,1,GSEC9.dta,GSEC9,1013000213,1.0,1.0,1.0,4.0,96.0,1.0,...,NaN,NaN,1.0,8.0,8.0,1.0,2.0,5.0,8.0,1.0


Merged panel shape: (21239, 29)
Rows in base: 21239
Rows after merge: 21239
has_GSEC9
True     17789
False     3450
Name: count, dtype: int64


,Wave,HHID,hh_id_obs,GSEC9__SOURCE_FILE,GSEC9__SOURCE_SECTION,GSEC9__H9Q1,GSEC9__H9Q2,GSEC9__H9Q3,GSEC9__H9Q4,GSEC9__H9Q5,...,GSEC9__H9Q14,GSEC9__H9Q16,GSEC9__H9Q17,GSEC9__H9Q18,GSEC9__H9Q19,GSEC9__H9Q20,GSEC9__H9Q21,GSEC9__H9Q22,GSEC9__H9Q23,has_GSEC9
0,1,1013000201,7000043,GSEC9.dta,GSEC9,1.0,1.0,2.0,4.0,3.0,...,NaN,9.0,2.0,2.0,1.0,3.0,2.0,5.0,1.0,True
1,1,1013000204,7000045,GSEC9.dta,GSEC9,1.0,1.0,1.0,4.0,3.0,...,NaN,2.0,2.0,2.0,1.0,2.0,5.0,2.0,1.0,True
2,1,1013000206,7000046,GSEC9.dta,GSEC9,2.0,5.0,1.0,4.0,6.0,...,1500.0,9.0,2.0,2.0,1.0,2.0,5.0,4.0,1.0,True
3,1,1013000210,7000047,GSEC9.dta,GSEC9,1.0,1.0,2.0,4.0,3.0,...,NaN,1.0,8.0,NaN,NaN,1.0,5.0,2.0,1.0,True
4,1,1013000213,7000049,GSEC9.dta,GSEC9,1.0,1.0,1.0,4.0,96.0,...,NaN,1.0,8.0,8.0,1.0,2.0,5.0,8.0,1.0,True


In [3]:
### GSEC10 + 12

def merge_hh_section(panel, section_file, section_name):

    sec = load_table_clean(section_file)
    sec = standardize_keys(sec)

    key_cols = ["Wave", "HHID"]

    duplicates_temp = sec[sec.duplicated(key_cols, keep=False)].copy()

    if len(duplicates_temp) > 0:
        print(f"\nWARNING: {section_name} has duplicate Wave-HHID rows.")
        print(f"Duplicated rows moved to duplicates_temp: {len(duplicates_temp)}")

        display(
            duplicates_temp
            .sort_values(key_cols)
            .head(30)
        )

        sec = sec.drop_duplicates(key_cols, keep=False).copy()

    else:
        print(f"{section_name}: no duplicate Wave-HHID rows.")

    check_unique_keys(sec, key_cols, section_name)

    sec = sec.rename(
        columns={
            col: f"{section_name}__{col}"
            for col in sec.columns
            if col not in key_cols
        }
    )

    before_rows = len(panel)

    panel = panel.merge(
        sec,
        on=key_cols,
        how="left",
        validate="1:1"
    )

    after_rows = len(panel)

    if before_rows != after_rows:
        raise ValueError(
            f"Row count changed when merging {section_name}: "
            f"{before_rows} -> {after_rows}"
        )

    section_vars = [
        c for c in panel.columns
        if c.startswith(f"{section_name}__")
    ]

    panel[f"has_{section_name}"] = panel[section_vars].notna().any(axis=1)

    print(f"\n{section_name} merged.")
    print("Clean section shape:", sec.shape)
    print("Panel shape:", panel.shape)
    print(panel[f"has_{section_name}"].value_counts(dropna=False))

    return panel, duplicates_temp

In [4]:
# duplicates function test

all_duplicates = []

for section_name, section_file in {
    "GSEC10": GSEC10_FILE,
    "GSEC12": GSEC12_FILE,
    "GSEC13": GSEC13_FILE,
    "GSEC17": GSEC17_FILE,
}.items():

    hh_panel, dup = merge_hh_section(
        panel=hh_panel,
        section_file=section_file,
        section_name=section_name
    )

    if len(dup) > 0:
        dup["section"] = section_name
        all_duplicates.append(dup)

duplicates_temp = (
    pd.concat(all_duplicates, ignore_index=True)
    if len(all_duplicates) > 0
    else pd.DataFrame()
)


Duplicated rows moved to duplicates_temp: 1013


,Wave,SOURCE_FILE,SOURCE_SECTION,HHID,H10Q1,H10Q6,H10Q09
15,1,GSEC10A.dta,GSEC10A,1.021E+11,1,2,6
28,1,GSEC10A.dta,GSEC10A,1.021E+11,2,2,6
29,1,GSEC10A.dta,GSEC10A,1.021E+11,2,2,6
31,1,GSEC10A.dta,GSEC10A,1.021E+11,2,2,8
46,1,GSEC10A.dta,GSEC10A,1.021E+11,1,2,6
54,1,GSEC10A.dta,GSEC10A,1.021E+11,2,2,3
57,1,GSEC10A.dta,GSEC10A,1.021E+11,2,2,8
72,1,GSEC10A.dta,GSEC10A,1.021E+11,2,2,4
76,1,GSEC10A.dta,GSEC10A,1.021E+11,NaN,NaN,NaN
77,1,GSEC10A.dta,GSEC10A,1.021E+11,1,2,6


GSEC10: keys are unique.

GSEC10 merged.
Clean section shape: (20125, 7)
Panel shape: (21239, 36)
has_GSEC10
True     19931
False     1308
Name: count, dtype: int64
GSEC12: no duplicate Wave-HHID rows.
GSEC12: keys are unique.

GSEC12 merged.
Clean section shape: (15903, 5)
Panel shape: (21239, 40)
has_GSEC12
True     15900
False     5339
Name: count, dtype: int64
GSEC13: no duplicate Wave-HHID rows.
GSEC13: keys are unique.

GSEC13 merged.
Clean section shape: (2941, 9)
Panel shape: (21239, 48)
has_GSEC13
False    18299
True      2940
Name: count, dtype: int64

Duplicated rows moved to duplicates_temp: 1002


,Wave,SOURCE_FILE,SOURCE_SECTION,HHID,H17Q1,H17Q2,H17Q3,H17Q4,H17Q5,H17Q7,H17Q8,H17Q9,H17Q10,H17Q11
15,1,GSEC17.dta,GSEC17,1.021E+11,2,1,2,1,3,4,4,2,NaN,NaN
28,1,GSEC17.dta,GSEC17,1.021E+11,1,1,2,1,3,4,4,2,NaN,NaN
29,1,GSEC17.dta,GSEC17,1.021E+11,1,1,2,1,3,7,7,2,NaN,NaN
31,1,GSEC17.dta,GSEC17,1.021E+11,1,1,2,1,3,12,96,2,NaN,NaN
46,1,GSEC17.dta,GSEC17,1.021E+11,1,1,1,1,4,6,12,2,NaN,NaN
54,1,GSEC17.dta,GSEC17,1.021E+11,1,1,3,1,2,12,12,1,IJ,E
57,1,GSEC17.dta,GSEC17,1.021E+11,1,1,1,1,2,4,12,2,NaN,NaN
72,1,GSEC17.dta,GSEC17,1.021E+11,2,1,2,2,3,1,1,2,NaN,NaN
76,1,GSEC17.dta,GSEC17,1.021E+11,2,1,1,1,2,2,12,2,NaN,NaN
77,1,GSEC17.dta,GSEC17,1.021E+11,2,1,1,1,3,2,12,1,F,E


GSEC17: keys are unique.

GSEC17 merged.
Clean section shape: (16880, 14)
Panel shape: (21239, 61)
has_GSEC17
True     16752
False     4487
Name: count, dtype: int64


In [21]:
# intermediate save modiule

OUT = ROOT / "Built panels"
OUT.mkdir(exist_ok=True)

hh_panel.to_excel(OUT / "hh_panel_roster_9_10_11_12_13_17_18.xlsx", index=False)
hh_panel.to_parquet(OUT / "hh_panel_roster_9_10_11_12_13_17_18.parquet", index=False)

"""
save duplicates
gsec16_duplicates.to_excel(
OUT / "gsec16_duplicates.xlsx",
index=False
)
"""

In [6]:
# GSEC11: reshape income-code index rows wide by Wave-HHID
#=========================================================================

GSEC11_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC11_standardized.csv"
)

def clean_code(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x == "" or x.lower() == "nan":
        return pd.NA

    if x.endswith(".0"):
        x = x[:-2]

    return x


def to_number(s):
    return pd.to_numeric(s, errors="coerce")


gsec11 = load_table_clean(GSEC11_FILE)
gsec11 = standardize_keys(gsec11)

key_cols = ["Wave", "HHID"]

gsec11["H11Q03_code"] = gsec11["H11Q03"].apply(clean_code)

gsec11["H11AQ04_num"] = to_number(gsec11["H11AQ04"]).fillna(0)
gsec11["H11AQ05_num"] = to_number(gsec11["H11AQ05"]).fillna(0)
gsec11["H11AQ06_num"] = to_number(gsec11["H11AQ06"]).fillna(0)

gsec11["H11AQ_inc_value"] = (
    gsec11["H11AQ05_num"]
    + gsec11["H11AQ06_num"]
)

gsec11_q01_conflicts = (
    gsec11
    .groupby(key_cols)["H11Q01"]
    .nunique(dropna=True)
    .reset_index(name="n_H11Q01_values")
    .query("n_H11Q01_values > 1")
)

print("H11Q01 conflicts:", len(gsec11_q01_conflicts))
display(gsec11_q01_conflicts.head(20))

# Keep H11Q01 once per household-wave.
# If repeated, this takes the first non-missing value and leaves conflicts above for review.
gsec11_q01 = (
    gsec11
    .sort_values(key_cols)
    .groupby(key_cols, as_index=False)["H11Q01"]
    .agg(lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else pd.NA)
)

gsec11_duplicate_income_codes = (
    gsec11
    .dropna(subset=["H11Q03_code"])
    .groupby(key_cols + ["H11Q03_code"])
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
)

print("Repeated Wave-HHID-income code groups:", len(gsec11_duplicate_income_codes))
display(gsec11_duplicate_income_codes.head(20))

gsec11_by_code = (
    gsec11
    .dropna(subset=["H11Q03_code"])
    .groupby(key_cols + ["H11Q03_code"], as_index=False)
    .agg(
        H11AQ04_dummy=("H11AQ04_num", "max"),
        H11AQ_inc_value=("H11AQ_inc_value", "sum")
    )
)

income_codes = sorted(
    gsec11_by_code["H11Q03_code"].dropna().unique(),
    key=lambda x: int(x)
)

print("Observed H11Q03 income codes:", income_codes)

gsec11_dummies = (
    gsec11_by_code
    .pivot(index=key_cols, columns="H11Q03_code", values="H11AQ04_dummy")
    .reindex(columns=income_codes)
    .fillna(0)
)

gsec11_dummies.columns = [
    f"H11Q03({code})"
    for code in gsec11_dummies.columns
]

gsec11_values = (
    gsec11_by_code
    .pivot(index=key_cols, columns="H11Q03_code", values="H11AQ_inc_value")
    .reindex(columns=income_codes)
    .fillna(0)
)

gsec11_values.columns = [
    f"H11AQ_inc_value({code})"
    for code in gsec11_values.columns
]

gsec11_wide = (
    gsec11_q01
    .merge(gsec11_dummies.reset_index(), on=key_cols, how="left")
    .merge(gsec11_values.reset_index(), on=key_cols, how="left")
)

wide_cols = [c for c in gsec11_wide.columns if c not in key_cols + ["H11Q01"]]
gsec11_wide[wide_cols] = gsec11_wide[wide_cols].fillna(0)

# turns structural zeros into missing values

for code in income_codes:
    dummy_col = f"H11Q03({code})"
    value_col = f"H11AQ_inc_value({code})"

    if dummy_col in gsec11_wide.columns and value_col in gsec11_wide.columns:
        gsec11_wide.loc[gsec11_wide[dummy_col] == 0, value_col] = pd.NA

dummy_cols = [c for c in gsec11_wide.columns if c.startswith("H11Q03(")]
gsec11_wide = gsec11_wide.drop(columns=dummy_cols)

dummy_cols = [c for c in gsec11_wide.columns if c.startswith("H11Q03(")]
gsec11_wide[dummy_cols] = gsec11_wide[dummy_cols].astype("Int64")

check_unique_keys(gsec11_wide, key_cols, "GSEC11 wide")

print("Original GSEC11 shape:", gsec11.shape)
print("Wide GSEC11 shape:", gsec11_wide.shape)

display(gsec11_wide.head())

H11Q01 conflicts: 0


,Wave,HHID,n_H11Q01_values


Repeated Wave-HHID-income code groups: 2


,Wave,HHID,H11Q03_code,n_rows
10221,2,4073000306,13,2
10222,2,4073000306,45,2


Observed H11Q03 income codes: ['11', '12', '13', '21', '22', '23', '31', '32', '33', '34', '35', '36', '41', '42', '43', '44', '45', '48']
GSEC11 wide: keys are unique.
Original GSEC11 shape: (29243, 14)
Wide GSEC11 shape: (21027, 21)


,Wave,HHID,H11Q01,H11AQ_inc_value(11),H11AQ_inc_value(12),H11AQ_inc_value(13),H11AQ_inc_value(21),H11AQ_inc_value(22),H11AQ_inc_value(23),H11AQ_inc_value(31),...,H11AQ_inc_value(33),H11AQ_inc_value(34),H11AQ_inc_value(35),H11AQ_inc_value(36),H11AQ_inc_value(41),H11AQ_inc_value(42),H11AQ_inc_value(43),H11AQ_inc_value(44),H11AQ_inc_value(45),H11AQ_inc_value(48)
0,1,1013000201,4.0,NaN,NaN,4320000.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1013000204,4.0,NaN,NaN,1440000.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1013000206,4.0,NaN,NaN,NaN,4800000.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1013000210,4.0,NaN,NaN,5250000.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,1013000213,3.0,NaN,400000.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
 # GSEC16 reshape

GSEC16_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC16_standardized.csv"
)

def clean_code(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x == "" or x.lower() == "nan":
        return pd.NA

    if x.endswith(".0"):
        x = x[:-2]

    return x


def is_yes(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()
    return s in ["1", "1.0", "YES", "Y", "TRUE"]


def first_nonmissing(s):
    s = s.dropna()

    if len(s) == 0:
        return pd.NA

    return s.iloc[0]


gsec16 = load_table_clean(GSEC16_FILE)
gsec16 = standardize_keys(gsec16)

key_cols = ["Wave", "HHID"]

gsec16["H16Q00_code"] = gsec16["H16Q00"].apply(clean_code)

before_rows = len(gsec16)

gsec16_yes = gsec16[
    gsec16["H16Q00_code"].notna()
    & gsec16["H16Q1"].map(is_yes)
].copy()

print(f"GSEC16: dropped {before_rows - len(gsec16_yes):,} non-yes / empty shock rows")
print(f"GSEC16: kept {len(gsec16_yes):,} yes shock rows")

shock_codes = [
    "101", "1011", "1012",
    "102",
    "103", "1031", "1032",
    "104", "105", "106", "107", "108", "109",
    "110", "111", "112", "113", "114", "115",
    "116", "117", "118",
]

observed_codes = sorted(
    gsec16_yes["H16Q00_code"].dropna().unique(),
    key=lambda x: int(x)
)

unexpected_codes = [c for c in observed_codes if c not in shock_codes]

print("Observed shock codes:", observed_codes)
print("Unexpected shock codes:", unexpected_codes)

value_cols = [
    "H16Q2A", "H16Q2B",
    "H16Q4A", "H16Q4B", "H16Q4C",
]

dummy_source_cols = [
    "H16Q1",
    "H16Q3A", "H16Q3B", "H16Q3C", "H16Q3D",
]

gsec16_duplicates = (
    gsec16_yes
    .groupby(key_cols + ["H16Q00_code"])
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
)

print("Repeated Wave-HHID-shock-code groups:", len(gsec16_duplicates))
display(gsec16_duplicates.head(20))

for col in dummy_source_cols:
    gsec16_yes[f"{col}_dummy"] = gsec16_yes[col].map(lambda x: 1 if is_yes(x) else 0)

for col in value_cols:
    gsec16_yes[col] = pd.to_numeric(gsec16_yes[col], errors="coerce")

agg_spec = {
    "H16Q1_dummy": "max",
    "H16Q3A_dummy": "max",
    "H16Q3B_dummy": "max",
    "H16Q3C_dummy": "max",
    "H16Q3D_dummy": "max",
    "H16Q2A": first_nonmissing,
    "H16Q2B": first_nonmissing,
    "H16Q4A": first_nonmissing,
    "H16Q4B": first_nonmissing,
    "H16Q4C": first_nonmissing,
}

gsec16_by_code = (
    gsec16_yes
    .groupby(key_cols + ["H16Q00_code"], as_index=False)
    .agg(agg_spec)
)

wide_pieces = []

reshape_specs = [
    ("H16Q1_dummy", "H16Q1", True),
    ("H16Q2A", "H16Q2A", False),
    ("H16Q2B", "H16Q2B", False),
    ("H16Q3A_dummy", "H16Q3A", True),
    ("H16Q3B_dummy", "H16Q3B", True),
    ("H16Q3C_dummy", "H16Q3C", True),
    ("H16Q3D_dummy", "H16Q3D", True),
    ("H16Q4A", "H16Q4A", False),
    ("H16Q4B", "H16Q4B", False),
    ("H16Q4C", "H16Q4C", False),
]

for source_col, output_prefix, is_dummy_col in reshape_specs:

    wide_part = (
        gsec16_by_code
        .pivot(index=key_cols, columns="H16Q00_code", values=source_col)
        .reindex(columns=shock_codes)
    )

    wide_part.columns = [
        f"{output_prefix}({code})"
        for code in wide_part.columns
    ]

    if is_dummy_col:
        wide_part = wide_part.fillna(0).astype("Int64")

    wide_pieces.append(wide_part)

gsec16_wide = pd.concat(wide_pieces, axis=1).reset_index()

check_unique_keys(gsec16_wide, key_cols, "GSEC16 wide")

print("Original GSEC16 shape:", gsec16.shape)
print("Yes-only GSEC16 shape:", gsec16_yes.shape)
print("Wide GSEC16 shape:", gsec16_wide.shape)

display(gsec16_wide.head())


GSEC16: dropped 2,501 non-yes / empty shock rows
GSEC16: kept 10,896 yes shock rows
Observed shock codes: ['101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '1011', '1012', '1031', '1032']
Unexpected shock codes: []
Repeated Wave-HHID-shock-code groups: 104


,Wave,HHID,H16Q00_code,n_rows
0,1,1.02E+11,101,2
1,1,1.02E+11,110,2
9,1,1.05E+11,101,5
10,1,1.05E+11,114,3
12,1,1.06E+11,101,4
13,1,1.06E+11,104,2
14,1,1.06E+11,105,2
17,1,1.06E+11,111,2
18,1,1.07E+11,101,3
20,1,1.07E+11,110,3


GSEC16 wide: keys are unique.
Original GSEC16 shape: (13397, 16)
Yes-only GSEC16 shape: (10896, 21)
Wide GSEC16 shape: (8034, 222)


,Wave,HHID,H16Q1(101),H16Q1(1011),H16Q1(1012),H16Q1(102),H16Q1(103),H16Q1(1031),H16Q1(1032),H16Q1(104),...,H16Q4C(109),H16Q4C(110),H16Q4C(111),H16Q4C(112),H16Q4C(113),H16Q4C(114),H16Q4C(115),H16Q4C(116),H16Q4C(117),H16Q4C(118)
0,1,1.02E+11,1,0,0,0,0,0,0,0,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
1,1,1.03E+11,1,0,0,0,0,0,0,0,...,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1.04E+11,1,0,0,0,0,0,0,0,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1.05E+11,1,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN
4,1,1.06E+11,1,0,0,0,0,0,0,1,...,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
GSEC18_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC18_standardized.csv"
)

def clean_letter(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().upper()

    if x == "" or x.lower() == "nan":
        return pd.NA

    return x


def is_yes(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()
    return s in ["1", "1.0", "YES", "Y", "TRUE"]


def first_nonmissing(s):
    s = s.dropna()

    if len(s) == 0:
        return pd.NA

    return s.iloc[0]


gsec18 = load_table_clean(GSEC18_FILE)
gsec18 = standardize_keys(gsec18)

key_cols = ["Wave", "HHID"]

gsec18["H18Q1_letter"] = gsec18["H18Q1"].apply(clean_letter)
gsec18["H18Q9_letter"] = gsec18["H18Q9"].apply(clean_letter)

letters_q1 = ["A", "B", "C", "D"]
letters_q9 = ["A", "B", "C", "D", "E", "F"]

gsec18_q1 = gsec18[
    gsec18["H18Q1_letter"].isin(letters_q1)
    & gsec18["H18Q2"].map(is_yes)
].copy()

gsec18_q1["H18Q1_dummy"] = 1
gsec18_q1["H18Q5_dummy"] = gsec18_q1["H18Q5"].map(lambda x: 1 if is_yes(x) else 0)
gsec18_q1["H18Q4"] = pd.to_numeric(gsec18_q1["H18Q4"], errors="coerce")

q1_dupes = (
    gsec18_q1
    .groupby(key_cols + ["H18Q1_letter"])
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
)

print("Repeated Wave-HHID-H18Q1 letter groups:", len(q1_dupes))
display(q1_dupes.head(20))

gsec18_q1_by_letter = (
    gsec18_q1
    .groupby(key_cols + ["H18Q1_letter"], as_index=False)
    .agg(
        H18Q1_dummy=("H18Q1_dummy", "max"),
        H18Q4=("H18Q4", first_nonmissing),
        H18Q5_dummy=("H18Q5_dummy", "max"),
    )
)

q1_pieces = []

for source_col, output_prefix, is_dummy_col in [
    ("H18Q1_dummy", "H18Q1", True),
    ("H18Q4", "H18Q4", False),
    ("H18Q5_dummy", "H18Q5", True),
]:

    wide_part = (
        gsec18_q1_by_letter
        .pivot(index=key_cols, columns="H18Q1_letter", values=source_col)
        .reindex(columns=letters_q1)
    )

    wide_part.columns = [
        f"{output_prefix}{letter}"
        for letter in wide_part.columns
    ]

    if is_dummy_col:
        wide_part = wide_part.fillna(0).astype("Int64")

    q1_pieces.append(wide_part)

gsec18_q1_wide = pd.concat(q1_pieces, axis=1).reset_index()

gsec18_q9 = gsec18[
    gsec18["H18Q9_letter"].isin(letters_q9)
    & gsec18["H18Q10"].map(is_yes)
].copy()

gsec18_q9["H18Q9_dummy"] = 1
gsec18_q9["H18Q11"] = pd.to_numeric(gsec18_q9["H18Q11"], errors="coerce")

q9_dupes = (
    gsec18_q9
    .groupby(key_cols + ["H18Q9_letter"])
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
)

print("Repeated Wave-HHID-H18Q9 letter groups:", len(q9_dupes))
display(q9_dupes.head(20))

gsec18_q9_by_letter = (
    gsec18_q9
    .groupby(key_cols + ["H18Q9_letter"], as_index=False)
    .agg(
        H18Q9_dummy=("H18Q9_dummy", "max"),
        H18Q11=("H18Q11", first_nonmissing),
    )
)

q9_pieces = []

for source_col, output_prefix, is_dummy_col in [
    ("H18Q9_dummy", "H18Q9", True),
    ("H18Q11", "H18Q11", False),
]:

    wide_part = (
        gsec18_q9_by_letter
        .pivot(index=key_cols, columns="H18Q9_letter", values=source_col)
        .reindex(columns=letters_q9)
    )

    wide_part.columns = [
        f"{output_prefix}{letter}"
        for letter in wide_part.columns
    ]

    if is_dummy_col:
        wide_part = wide_part.fillna(0).astype("Int64")

    q9_pieces.append(wide_part)

gsec18_q9_wide = pd.concat(q9_pieces, axis=1).reset_index()

gsec18_wide = gsec18_q1_wide.merge(
    gsec18_q9_wide,
    on=key_cols,
    how="outer",
    validate="1:1"
)

check_unique_keys(gsec18_wide, key_cols, "GSEC18 wide")

print("Original GSEC18 shape:", gsec18.shape)
print("Q1 block shape:", gsec18_q1_wide.shape)
print("Q9 block shape:", gsec18_q9_wide.shape)
print("Wide GSEC18 shape:", gsec18_wide.shape)

display(gsec18_wide.head())

Repeated Wave-HHID-H18Q1 letter groups: 0


,Wave,HHID,H18Q1_letter,n_rows


Repeated Wave-HHID-H18Q9 letter groups: 0


,Wave,HHID,H18Q9_letter,n_rows


GSEC18 wide: keys are unique.
Original GSEC18 shape: (28338, 13)
Q1 block shape: (8306, 14)
Q9 block shape: (3465, 14)
Wide GSEC18 shape: (8357, 26)


,Wave,HHID,H18Q1A,H18Q1B,H18Q1C,H18Q1D,H18Q4A,H18Q4B,H18Q4C,H18Q4D,...,H18Q9C,H18Q9D,H18Q9E,H18Q9F,H18Q11A,H18Q11B,H18Q11C,H18Q11D,H18Q11E,H18Q11F
0,1,1013000206,0,0,1,1,NaN,NaN,5.0,0.0,...,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1013000213,0,0,0,1,NaN,NaN,NaN,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,5.0,NaN
2,1,101300021302,0,1,1,1,NaN,0.0,0.0,0.0,...,0,0,1,0,NaN,NaN,NaN,NaN,4.0,NaN
3,1,1021000102,1,0,1,1,1.0,NaN,5.0,7.0,...,0,1,1,0,4.0,2.0,NaN,3.0,4.0,NaN
4,1,1021000108,1,0,1,1,15.0,NaN,5.0,5.0,...,1,0,0,0,NaN,2.0,3.0,NaN,NaN,NaN


In [10]:
GSEC15C_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC15C_standardized.csv"
)

def clean_item_code(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x == "" or x.lower() == "nan":
        return pd.NA

    if x.endswith(".0"):
        x = x[:-2]

    return x


def to_number(s):
    return pd.to_numeric(s, errors="coerce")


def item_code_sort_key(code):
    code = str(code)

    base = code.split("_")[0]
    suffix = code.split("_")[1] if "_" in code else ""

    base_num = int(base) if base.isdigit() else 999999
    suffix_num = int(suffix) if suffix.isdigit() else 0

    return (base_num, suffix_num, code)


def slot_value(value_col, qty_col, price_col):
    value = to_number(gsec15c[value_col])
    qty = to_number(gsec15c[qty_col])
    price = to_number(gsec15c[price_col])

    derived = qty * price

    return value.where(value.notna(), derived)


gsec15c = load_table_clean(GSEC15C_FILE)
gsec15c = standardize_keys(gsec15c)

key_cols = ["Wave", "HHID"]

gsec15c["ITEM_CODE_clean"] = gsec15c["ITEM_CODE"].apply(clean_item_code)

target_base_codes = (
    list(range(301, 312))
    + list(range(451, 460))
    + list(range(461, 470))
    + list(range(501, 506))
    + list(range(601, 606))
)

target_base_codes = [str(c) for c in target_base_codes]

gsec15c["ITEM_CODE_base"] = (
    gsec15c["ITEM_CODE_clean"]
    .astype("string")
    .str.split("_")
    .str[0]
)

gsec15c_target = gsec15c[
    gsec15c["ITEM_CODE_clean"].notna()
    & gsec15c["ITEM_CODE_base"].isin(target_base_codes)
].copy()

excluded_codes = sorted(
    set(gsec15c["ITEM_CODE_clean"].dropna().unique())
    - set(gsec15c_target["ITEM_CODE_clean"].dropna().unique()),
    key=item_code_sort_key
)

print("Excluded non-target item codes:", excluded_codes)

gsec15c = gsec15c_target.copy()

gsec15c["H15C_slot1_value"] = slot_value("H15CQ5", "H15CQ4", "H15CQ10")
gsec15c["H15C_slot2_value"] = slot_value("H15CQ7", "H15CQ6", "H15CQ10")
gsec15c["H15C_slot3_value"] = slot_value("H15CQ9", "H15CQ8", "H15CQ10")

slot_cols = [
    "H15C_slot1_value",
    "H15C_slot2_value",
    "H15C_slot3_value",
]

gsec15c["H15C_item_value"] = gsec15c[slot_cols].sum(axis=1, min_count=1)

# Drop rows where the item code exists but no value can be reported or derived.
before_value_filter = len(gsec15c)

gsec15c_value_rows = gsec15c[
    gsec15c["H15C_item_value"].notna()
].copy()

print(
    "Dropped item-code rows with no reported or derived value:",
    before_value_filter - len(gsec15c_value_rows)
)

gsec15c_by_code = (
    gsec15c_value_rows
    .groupby(key_cols + ["ITEM_CODE_clean"], as_index=False)
    .agg(H15C_item_value=("H15C_item_value", "sum"))
)

item_codes = sorted(
    gsec15c_by_code["ITEM_CODE_clean"].dropna().unique(),
    key=item_code_sort_key
)

print("Included GSEC15C item codes:", item_codes)

gsec15c_wide = (
    gsec15c_by_code
    .pivot(index=key_cols, columns="ITEM_CODE_clean", values="H15C_item_value")
    .reindex(columns=item_codes)
)

gsec15c_wide.columns = [
    f"H15C({code})"
    for code in gsec15c_wide.columns
]

gsec15c_wide = gsec15c_wide.reset_index()

check_unique_keys(gsec15c_wide, key_cols, "GSEC15C wide")

print("Original GSEC15C shape:", load_table_clean(GSEC15C_FILE).shape)
print("Target-code rows:", gsec15c_target.shape)
print("Value rows:", gsec15c_value_rows.shape)
print("Wide GSEC15C shape:", gsec15c_wide.shape)

display(gsec15c_wide.head())

Excluded non-target item codes: ['108', '109', '312_1', '450_1', '460', '470', '482', '506', '507', '606_1', '109999', '4651999']
Dropped item-code rows with no reported or derived value: 83
Included GSEC15C item codes: ['301', '302', '303', '304', '305', '305_1', '305_2', '306', '307', '308', '309', '310', '311', '311_1', '311_2', '451', '451_1', '452', '452_1', '453', '454', '454_1', '454_2', '455', '456', '457', '458', '459', '459_1', '461', '462', '462_1', '462_2', '462_3', '462_4', '463', '464', '465', '465_1', '466', '466_1', '467', '467_1', '467_2', '468', '469', '501', '502', '503', '504', '505', '601', '602', '603', '604', '605']
GSEC15C wide: keys are unique.
Original GSEC15C shape: (219845, 13)
Target-code rows: (211437, 15)
Value rows: (211354, 19)
Wide GSEC15C shape: (20211, 58)


,Wave,HHID,H15C(301),H15C(302),H15C(303),H15C(304),H15C(305),H15C(305_1),H15C(305_2),H15C(306),...,H15C(501),H15C(502),H15C(503),H15C(504),H15C(505),H15C(601),H15C(602),H15C(603),H15C(604),H15C(605)
0,1,1.021E+11,665000.0,635000.0,70000.0,22200.0,188100.0,NaN,NaN,82000.0,...,NaN,103200.0,56000.0,NaN,NaN,5500.0,NaN,8000.0,339000.0,NaN
1,1,1.033E+11,NaN,8000.0,NaN,20000.0,9000.0,NaN,NaN,NaN,...,NaN,9000.0,NaN,NaN,NaN,NaN,NaN,NaN,11000.0,NaN
2,1,1.041E+11,15000.0,NaN,NaN,NaN,6000.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1.043E+11,45000.0,55000.0,NaN,NaN,54000.0,NaN,NaN,6000.0,...,NaN,99000.0,NaN,NaN,NaN,NaN,NaN,NaN,10000.0,NaN
4,1,1.053E+11,140000.0,20000.0,NaN,4000.0,30000.0,NaN,NaN,NaN,...,NaN,27000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# merge hhpanel and wide sections

PANEL_FILE = (
    ROOT
    / "Built panels"
    / "HH_panel_with_GSEC9_10_12_13_17.xlsx"
)

hh_panel = load_table_clean(PANEL_FILE)
hh_panel = standardize_keys(hh_panel)

key_cols = ["Wave", "HHID"]


def key_diagnostics(left, right, right_name, key_cols=["Wave", "HHID"]):
    print(f"\n==============================")
    print(f"Diagnostics for {right_name}")
    print("==============================")

    left_dupes = left[left.duplicated(key_cols, keep=False)].copy()
    right_dupes = right[right.duplicated(key_cols, keep=False)].copy()

    print(f"Left panel rows: {len(left):,}")
    print(f"{right_name} rows: {len(right):,}")

    print(f"Left duplicate key rows: {len(left_dupes):,}")
    print(f"{right_name} duplicate key rows: {len(right_dupes):,}")

    if len(left_dupes) > 0:
        print("\nLeft duplicate keys:")
        display(left_dupes.sort_values(key_cols).head(20))

    if len(right_dupes) > 0:
        print(f"\n{right_name} duplicate keys:")
        display(right_dupes.sort_values(key_cols).head(20))

    left_keys = left[key_cols].drop_duplicates()
    right_keys = right[key_cols].drop_duplicates()

    matched = left_keys.merge(
        right_keys,
        on=key_cols,
        how="inner"
    )

    left_not_in_right = left_keys.merge(
        right_keys,
        on=key_cols,
        how="left",
        indicator=True
    ).query("_merge == 'left_only'").drop(columns="_merge")

    right_not_in_left = right_keys.merge(
        left_keys,
        on=key_cols,
        how="left",
        indicator=True
    ).query("_merge == 'left_only'").drop(columns="_merge")

    print(f"\nUnique left keys: {len(left_keys):,}")
    print(f"Unique {right_name} keys: {len(right_keys):,}")
    print(f"Matched keys: {len(matched):,}")
    print(f"Left keys missing {right_name}: {len(left_not_in_right):,}")
    print(f"{right_name} keys missing from left panel: {len(right_not_in_left):,}")

    print("\nMatched keys by wave:")
    display(
        matched
        .groupby("Wave")
        .size()
        .reset_index(name="matched_keys")
    )

    print(f"\n{right_name} keys missing from left panel by wave:")
    if len(right_not_in_left) > 0:
        display(
            right_not_in_left
            .groupby("Wave")
            .size()
            .reset_index(name="n_missing_from_left")
        )
        display(right_not_in_left.sort_values(key_cols).head(20))
    else:
        print("None")

    if len(right_dupes) > 0:
        raise ValueError(
            f"{right_name} has duplicate keys on {key_cols}. "
            "Fix before merging with validate='1:1'."
        )


def merge_wide_section(panel, section_wide, section_name, key_cols=["Wave", "HHID"]):
    section_wide = section_wide.copy()
    section_wide = standardize_keys(section_wide)

    key_diagnostics(panel, section_wide, section_name, key_cols)

    before_rows = len(panel)

    panel = panel.merge(
        section_wide,
        on=key_cols,
        how="left",
        validate="1:1"
    )

    after_rows = len(panel)

    if before_rows != after_rows:
        raise ValueError(
            f"Row count changed while merging {section_name}: "
            f"{before_rows:,} -> {after_rows:,}"
        )

    section_vars = [
        c for c in section_wide.columns
        if c not in key_cols
    ]

    panel[f"has_{section_name}"] = panel[section_vars].notna().any(axis=1)

    print(f"\n{section_name} merged.")
    print("Panel shape:", panel.shape)
    print(panel[f"has_{section_name}"].value_counts(dropna=False))

    return panel


check_unique_keys(hh_panel, key_cols, "HH panel before wide merges")

hh_panel = merge_wide_section(
    panel=hh_panel,
    section_wide=gsec11_wide,
    section_name="GSEC11_wide",
    key_cols=key_cols
)

hh_panel = merge_wide_section(
    panel=hh_panel,
    section_wide=gsec18_wide,
    section_name="GSEC18_wide",
    key_cols=key_cols
)

print("\nFinal panel shape:", hh_panel.shape)
display(hh_panel.head())

HH panel before wide merges: keys are unique.

Diagnostics for GSEC11_wide
Left panel rows: 21,239
GSEC11_wide rows: 21,027
Left duplicate key rows: 0
GSEC11_wide duplicate key rows: 0

Unique left keys: 21,239
Unique GSEC11_wide keys: 21,027
Matched keys: 21,020
Left keys missing GSEC11_wide: 219
GSEC11_wide keys missing from left panel: 7

Matched keys by wave:


,Wave,matched_keys
0,1,2939
1,2,2630
2,3,2801
3,4,3118
4,5,3301
5,7,3166
6,8,3065



GSEC11_wide keys missing from left panel by wave:


,Wave,n_missing_from_left
0,7,7


,Wave,HHID
14980,7,110f81c0d9254505838dcaf2497d130a
15271,7,27b67fdbc2714612b5230aff5b13d30b
16450,7,832a92a50ac84c27a6a44f11bbf7469b
16590,7,8de60b14dd494f88b1a99e48674c4e9b
16639,7,9147bc2f6725476eaa4bd9e36e701e80
17377,7,d10a687889de469687377204195f3db0
17570,7,e07bc322c4884559b4b8ca75c945dd3e



GSEC11_wide merged.
Panel shape: (21239, 81)
has_GSEC11_wide
True     21020
False      219
Name: count, dtype: int64

Diagnostics for GSEC18_wide
Left panel rows: 21,239
GSEC18_wide rows: 8,357
Left duplicate key rows: 0
GSEC18_wide duplicate key rows: 0

Unique left keys: 21,239
Unique GSEC18_wide keys: 8,357
Matched keys: 8,357
Left keys missing GSEC18_wide: 12,882
GSEC18_wide keys missing from left panel: 0

Matched keys by wave:


,Wave,matched_keys
0,1,2926
1,2,2635
2,3,2796



GSEC18_wide keys missing from left panel by wave:
None

GSEC18_wide merged.
Panel shape: (21239, 106)
has_GSEC18_wide
False    12882
True      8357
Name: count, dtype: int64

Final panel shape: (21239, 106)


,Wave,HHID,hh_id_obs,GSEC9__SOURCE_FILE,GSEC9__SOURCE_SECTION,GSEC9__H9Q1,GSEC9__H9Q2,GSEC9__H9Q3,GSEC9__H9Q4,GSEC9__H9Q5,...,H18Q9D,H18Q9E,H18Q9F,H18Q11A,H18Q11B,H18Q11C,H18Q11D,H18Q11E,H18Q11F,has_GSEC18_wide
0,1,1013000201,7000043,GSEC9.dta,GSEC9,1.0,1.0,2.0,4.0,3.0,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1,1013000204,7000045,GSEC9.dta,GSEC9,1.0,1.0,1.0,4.0,3.0,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1,1013000206,7000046,GSEC9.dta,GSEC9,2.0,5.0,1.0,4.0,6.0,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,True
3,1,1013000210,7000047,GSEC9.dta,GSEC9,1.0,1.0,2.0,4.0,3.0,...,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1,1013000213,7000049,GSEC9.dta,GSEC9,1.0,1.0,1.0,4.0,96.0,...,0,1,0,NaN,NaN,NaN,NaN,5.0,NaN,True


In [17]:
# recode LSMS 1/2 dummies to 1/0

yes_no_dummy_vars = [
    "H9Q12",
    "H9Q19",
    "H10Q1",
    "H10Q6",
    "H10Q09",
    "H12Q01",
    "H13Q14",
    "H13Q19",
    "H13Q20",
    "H13Q25",
    "H17Q2",
    "H17Q4",
    "H17Q9",
]


def recode_lsms_yes_no_once(panel, vars_to_recode):

    panel = panel.copy()

    report = []

    for var in vars_to_recode:
        matches = [
            col for col in panel.columns
            if col == var or col.endswith(f"__{var}")
        ]

        if len(matches) == 0:
            report.append({
                "variable": var,
                "matched_column": None,
                "status": "not found",
            })
            continue

        for col in matches:
            values = set(
                panel[col]
                .dropna()
                .astype(str)
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
                .unique()
            )

            # If 2 is gone and 0 is present, this column was probably already recoded.
            if "2" not in values and "0" in values:
                report.append({
                    "variable": var,
                    "matched_column": col,
                    "status": "skipped - already looks recoded",
                })
                continue

            before = panel[col].value_counts(dropna=False).to_dict()

            s = (
                panel[col]
                .astype("string")
                .str.strip()
                .str.replace(r"\.0$", "", regex=True)
            )

            panel[col] = pd.Series(pd.NA, index=panel.index, dtype="Int64")
            panel.loc[s == "1", col] = 1
            panel.loc[s == "2", col] = 0

            after = panel[col].value_counts(dropna=False).to_dict()

            report.append({
                "variable": var,
                "matched_column": col,
                "status": "recoded",
                "before_values": before,
                "after_values": after,
            })

    return panel, pd.DataFrame(report)

In [18]:
hh_panel = load_table_clean(PANEL_FILE)
hh_panel = standardize_keys(hh_panel)

hh_panel = merge_wide_section(
    panel=hh_panel,
    section_wide=gsec11_wide,
    section_name="GSEC11_wide",
    key_cols=["Wave", "HHID"]
)

hh_panel = merge_wide_section(
    panel=hh_panel,
    section_wide=gsec18_wide,
    section_name="GSEC18_wide",
    key_cols=["Wave", "HHID"]
)

yes_no_dummy_vars = [
    "H9Q12",
    "H9Q19",
    "H10Q1",
    "H10Q6",
    "H10Q09",
    "H12Q01",
    "H13Q14",
    "H13Q19",
    "H13Q20",
    "H13Q25",
    "H17Q2",
    "H17Q4",
    "H17Q9",
]

hh_panel, recode_report = recode_lsms_yes_no_once(
    panel=hh_panel,
    vars_to_recode=yes_no_dummy_vars
)

display(recode_report)


Diagnostics for GSEC11_wide
Left panel rows: 21,239
GSEC11_wide rows: 21,027
Left duplicate key rows: 0
GSEC11_wide duplicate key rows: 0

Unique left keys: 21,239
Unique GSEC11_wide keys: 21,027
Matched keys: 21,020
Left keys missing GSEC11_wide: 219
GSEC11_wide keys missing from left panel: 7

Matched keys by wave:


,Wave,matched_keys
0,1,2939
1,2,2630
2,3,2801
3,4,3118
4,5,3301
5,7,3166
6,8,3065



GSEC11_wide keys missing from left panel by wave:


,Wave,n_missing_from_left
0,7,7


,Wave,HHID
14980,7,110f81c0d9254505838dcaf2497d130a
15271,7,27b67fdbc2714612b5230aff5b13d30b
16450,7,832a92a50ac84c27a6a44f11bbf7469b
16590,7,8de60b14dd494f88b1a99e48674c4e9b
16639,7,9147bc2f6725476eaa4bd9e36e701e80
17377,7,d10a687889de469687377204195f3db0
17570,7,e07bc322c4884559b4b8ca75c945dd3e



GSEC11_wide merged.
Panel shape: (21239, 81)
has_GSEC11_wide
True     21020
False      219
Name: count, dtype: int64

Diagnostics for GSEC18_wide
Left panel rows: 21,239
GSEC18_wide rows: 8,357
Left duplicate key rows: 0
GSEC18_wide duplicate key rows: 0

Unique left keys: 21,239
Unique GSEC18_wide keys: 8,357
Matched keys: 8,357
Left keys missing GSEC18_wide: 12,882
GSEC18_wide keys missing from left panel: 0

Matched keys by wave:


,Wave,matched_keys
0,1,2926
1,2,2635
2,3,2796



GSEC18_wide keys missing from left panel by wave:
None

GSEC18_wide merged.
Panel shape: (21239, 106)
has_GSEC18_wide
False    12882
True      8357
Name: count, dtype: int64


,variable,matched_column,status,before_values,after_values
0,H9Q12,GSEC9__H9Q12,recoded,"{'2.0': 11822, '1.0': 5940, nan: 3476, '0.0': 1}","{0: 11822, 1: 5940, <NA>: 3477}"
1,H9Q19,GSEC9__H9Q19,recoded,"{nan: 9728, '1.0': 6693, '1': 2546, '2.0': 169...","{<NA>: 9728, 1: 9239, 0: 2272}"
2,H10Q1,GSEC10__H10Q1,recoded,"{'2': 17092, '1': 2832, nan: 1315}","{0: 17092, 1: 2832, <NA>: 1315}"
3,H10Q6,GSEC10__H10Q6,recoded,"{'2': 19814, nan: 1317, '1': 108}","{0: 19814, <NA>: 1317, 1: 108}"
4,H10Q09,GSEC10__H10Q09,recoded,"{'8': 14968, '6': 3804, nan: 1647, '5': 278, '...","{<NA>: 21139, 0: 55, 1: 45}"
5,H12Q01,GSEC12__H12Q01,recoded,"{nan: 7112, '1.0': 6934, '2.0': 5565, '2': 161...","{0: 7175, <NA>: 7112, 1: 6952}"
6,H13Q14,GSEC13__H13Q14,recoded,"{nan: 19570, '2.0': 1641, '1.0': 28}","{<NA>: 19570, 0: 1641, 1: 28}"
7,H13Q19,GSEC13__H13Q19,recoded,"{nan: 18307, '2.0': 2406, '1.0': 523, '7.0': 1...","{<NA>: 18310, 0: 2406, 1: 523}"
8,H13Q20,GSEC13__H13Q20,recoded,"{nan: 20703, '1.0': 447, '2.0': 89}","{<NA>: 20703, 1: 447, 0: 89}"
9,H13Q25,GSEC13__H13Q25,recoded,"{nan: 18306, '2.0': 2930, '1.0': 3}","{<NA>: 18306, 0: 2930, 1: 3}"


In [20]:
GSEC15A_FILE = (
    ROOT
    / "Finished sections"
    / "Household"
    / "GSEC15A_STANDARDIZED_standardized.csv"
)


def merge_section_before_prefix(panel, section_file, section_name, before_prefix):
    panel = panel.copy()

    key_cols = ["Wave", "HHID"]

    # Make rerunning the cell safe
    existing_section_cols = [
        c for c in panel.columns
        if c.startswith(f"{section_name}__") or c == f"has_{section_name}"
    ]

    if existing_section_cols:
        panel = panel.drop(columns=existing_section_cols)

    sec = load_table_clean(section_file)
    sec = standardize_keys(sec)

    check_unique_keys(sec, key_cols, section_name)

    missing_from_panel = (
        sec[key_cols]
        .drop_duplicates()
        .merge(panel[key_cols].drop_duplicates(), on=key_cols, how="left", indicator=True)
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
    )

    print(f"{section_name} rows:", len(sec))
    print(f"{section_name} keys missing from panel:", len(missing_from_panel))

    if len(missing_from_panel) > 0:
        display(missing_from_panel.groupby("Wave").size().reset_index(name="missing_keys"))
        display(missing_from_panel.head(20))

    sec = sec.rename(
        columns={
            col: f"{section_name}__{col}"
            for col in sec.columns
            if col not in key_cols
        }
    )

    before_rows = len(panel)

    panel = panel.merge(
        sec,
        on=key_cols,
        how="left",
        validate="1:1"
    )

    if len(panel) != before_rows:
        raise ValueError(f"Row count changed: {before_rows} -> {len(panel)}")

    section_cols = [
        c for c in panel.columns
        if c.startswith(f"{section_name}__")
    ]

    panel[f"has_{section_name}"] = panel[section_cols].notna().any(axis=1)

    section_block = section_cols + [f"has_{section_name}"]

    # Move GSEC15A block to immediately before the first GSEC9 column
    cols_without_section = [
        c for c in panel.columns
        if c not in section_block
    ]

    before_matches = [
        i for i, c in enumerate(cols_without_section)
        if c.startswith(before_prefix) or c == f"has_{before_prefix.rstrip('__')}"
    ]

    insert_at = before_matches[0] if before_matches else len(cols_without_section)

    new_cols = (
        cols_without_section[:insert_at]
        + section_block
        + cols_without_section[insert_at:]
    )

    panel = panel[new_cols]

    print(f"{section_name} merged.")
    print("Panel shape:", panel.shape)
    print(panel[f"has_{section_name}"].value_counts(dropna=False))

    return panel


hh_panel = merge_section_before_prefix(
    panel=hh_panel,
    section_file=GSEC15A_FILE,
    section_name="GSEC15A",
    before_prefix="GSEC9__"
)

GSEC15A: keys are unique.
GSEC15A rows: 21234
GSEC15A keys missing from panel: 66


,Wave,missing_keys
0,7,66


,Wave,HHID
6054,7,00c9353d8ebe42faabf5919b81d7fae7
6083,7,02dd448165ce46279ca601a02865d543
6092,7,037866653c7c4cb99a80f05a38cdafb2
6094,7,039a11571b874a88b7a6c200469fe4f3
6132,7,062da72d5d3a457e9336b62c8bb9096d
6200,7,0d0e29faff394154a69562b4527b48b8
6209,7,0e03e253c35d4333a1ffad2df9d38850
6243,7,110f81c0d9254505838dcaf2497d130a
6247,7,118e5671a1e0437baebdd563bf3e43df
6253,7,11edb121886741718afc9f6ee454ed07


GSEC15A merged.
Panel shape: (21239, 114)
has_GSEC15A
True     21168
False       71
Name: count, dtype: int64


In [22]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import pandas as pd

ROOT = Path(r"/")

AGSEC10_FILE = (
    ROOT
    / "Finished sections"
    / "Agriculture"
    / "AGSEC10_standardized.csv"
)

OUT = ROOT / "Finished sections" / "Agriculture"
OUT.mkdir(parents=True, exist_ok=True)


def clean_hhid_ag(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x == "":
        return pd.NA

    try:
        if "e" in x.lower():
            x = format(Decimal(x), "f")

        if x.endswith(".0"):
            x = x[:-2]

    except InvalidOperation:
        pass

    return x


def clean_wave(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().lower()
    x = x.replace("wave", "").replace("_", "").replace("-", "").strip()

    return int(float(x))


def is_yes(x):
    if pd.isna(x):
        return False

    s = str(x).strip().upper()
    return s in ["1", "1.0", "YES", "Y", "TRUE"]


def recode_service_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip().upper()

    if s == "" or s in ["NAN", "NONE", "<NA>", "."]:
        return pd.NA

    if s.endswith(".0"):
        s = s[:-2]

    if s in ["1", "2", "3", "4", "5", "6", "7"]:
        return s

    # Wave 1 string names
    service_map = {
        "NAADS": "1",
        "NATIONAL AGRICULTURAL ADVISORY SERVICES (NAADS)": "1",

        "INPUT SUPPLIER": "2",

        "NGO": "3",

        "COOPERATIVE": "4",
        "COOPERATIVE/FARMER'S ASSOCIATION": "4",
        "COOPERATIVE/FARMERS ASSOCIATION": "4",

        "LARGE SCALE FARMER": "5",

        "OTHERS": "6",
        "OTHER": "6",
        "OTHER (SPECIFY)": "6",
    }

    return service_map.get(s, pd.NA)


agsec10 = pd.read_csv(AGSEC10_FILE, dtype=str)

agsec10.columns = (
    agsec10.columns
    .str.strip()
    .str.upper()
)

agsec10 = agsec10.rename(columns={"WAVE": "Wave"})

agsec10["Wave"] = agsec10["Wave"].apply(clean_wave)
agsec10["HHID"] = agsec10["HHID"].apply(clean_hhid_ag)

agsec10["SERVICE_CODE"] = agsec10["EXT_SOURCE_ID"].apply(recode_service_id)

service_codes = ["1", "2", "3", "4", "5", "6", "7"]
key_cols = ["Wave", "HHID"]

unmapped_services = (
    agsec10[agsec10["SERVICE_CODE"].isna()]
    ["EXT_SOURCE_ID"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print("Unmapped EXT_SOURCE_ID values:")
display(unmapped_services)

agsec10_use = agsec10[
    agsec10["SERVICE_CODE"].isin(service_codes)
].copy()

agsec10_use["SEC10_SERVICE"] = agsec10_use["A10Q3"].map(lambda x: 1 if is_yes(x) else 0)

agsec10_use["SEC10_A"] = agsec10_use["A10Q5A"].map(lambda x: 1 if is_yes(x) else 0)
agsec10_use["SEC10_B"] = agsec10_use["A10Q5B"].map(lambda x: 1 if is_yes(x) else 0)
agsec10_use["SEC10_C"] = agsec10_use["A10Q5C"].map(lambda x: 1 if is_yes(x) else 0)

dup_check = (
    agsec10_use
    .groupby(key_cols + ["SERVICE_CODE"])
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
)

print("Duplicate Wave-HHID-service groups:", len(dup_check))
display(dup_check.head(20))

agsec10_by_service = (
    agsec10_use
    .groupby(key_cols + ["SERVICE_CODE"], as_index=False)
    .agg(
        SEC10_SERVICE=("SEC10_SERVICE", "max"),
        SEC10_A=("SEC10_A", "max"),
        SEC10_B=("SEC10_B", "max"),
        SEC10_C=("SEC10_C", "max"),
    )
)

wide_pieces = []

for source_col, prefix in [
    ("SEC10_SERVICE", "SEC10_SERVICE"),
    ("SEC10_A", "SEC10_A"),
    ("SEC10_B", "SEC10_B"),
    ("SEC10_C", "SEC10_C"),
]:

    wide_part = (
        agsec10_by_service
        .pivot(index=key_cols, columns="SERVICE_CODE", values=source_col)
        .reindex(columns=service_codes)
        .fillna(0)
        .astype("Int64")
    )

    wide_part.columns = [
        f"{prefix}_{code}"
        for code in wide_part.columns
    ]

    wide_pieces.append(wide_part)

agsec10_wide = pd.concat(wide_pieces, axis=1).reset_index()

dupes = agsec10_wide[agsec10_wide.duplicated(key_cols, keep=False)]

print("AGSEC10 wide shape:", agsec10_wide.shape)
print("Duplicate Wave-HHID rows after widening:", len(dupes))

if len(dupes) > 0:
    display(dupes.sort_values(key_cols).head(20))
    raise ValueError("AGSEC10 wide still has duplicate Wave-HHID rows.")

display(agsec10_wide.head())

agsec10_wide.to_csv(
    OUT / "AGSEC10_wide.csv",
    index=False
)

agsec10_wide.to_excel(
    OUT / "AGSEC10_wide_preview.xlsx",
    index=False
)


Unmapped EXT_SOURCE_ID values:


Series([], Name: EXT_SOURCE_ID, dtype: str)

Duplicate Wave-HHID-service groups: 31


,Wave,HHID,SERVICE_CODE,n_rows
28,1,105300000000,1,2
47,1,106300000000,1,2
256,1,207300000000,1,2
357,1,222300000000,1,3
385,1,302300000000,1,4
442,1,304100000000,1,6
481,1,307300000000,1,5
483,1,307300000000,3,2
588,1,317300000000,1,4
751,1,412300000000,1,2


AGSEC10 wide shape: (2973, 30)
Duplicate Wave-HHID rows after widening: 0


,Wave,HHID,SEC10_SERVICE_1,SEC10_SERVICE_2,SEC10_SERVICE_3,SEC10_SERVICE_4,SEC10_SERVICE_5,SEC10_SERVICE_6,SEC10_SERVICE_7,SEC10_A_1,...,SEC10_B_5,SEC10_B_6,SEC10_B_7,SEC10_C_1,SEC10_C_2,SEC10_C_3,SEC10_C_4,SEC10_C_5,SEC10_C_6,SEC10_C_7
0,1,1013000210,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,102100000000,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1021000113,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1021002003,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1021002501,1,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
